# CLEETS-SMART Quarterly EV Forecasting Notebook — corrected accuracy workflow

This corrected Colab notebook avoids the dependency issues caused by `scikit-learn`, `scipy`, and TensorFlow version conflicts. It uses stable `numpy`/`pandas` forecasting models, evaluates them with quarterly walk-forward backtesting, and selects the best-performing approach for EV keepership and EV charger forecasts.

Main changes:
1. Keeps quarterly observations instead of collapsing to annual rows.
2. Uses a clean quarterly time index.
3. Avoids broken `sklearn`/`scipy` imports.
4. Compares models using MAE, RMSE, and R².
5. Uses scenario carrying capacity `K` only where it is statistically meaningful: long-term bounded logistic forecasts.
6. Produces corrected forecasts to 2045.


In [ ]:
# ============================================================
# 0. Lightweight setup
# ============================================================

# Avoid force-reinstalling core Colab packages such as numpy, scipy, pandas, torch.
# This prevents the dependency conflicts you encountered earlier.

!pip install -q rdflib gdown plotly openpyxl

import re
import math
import unicodedata
from urllib.parse import quote

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import plotly.express as px

from rdflib import Graph, Namespace, RDF, RDFS, OWL, XSD, Literal

import gdown
from google.colab import files
import google.colab.data_table

google.colab.data_table.enable_dataframe_formatter()

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)

SEED = 42
np.random.seed(SEED)

print("Libraries loaded successfully.")
print("numpy:", np.__version__)
print("pandas:", pd.__version__)


# 1. Download and load source datasets

In [ ]:
KEEPERSHIP_URL = "https://drive.google.com/file/d/1o5kQicaRDBe8xP_uQMOZ9cBe2TU-r-HM/view?usp=sharing"
CHARGER_URL = "https://drive.google.com/file/d/1-HjLsZAvcDyYHgn2jRwIVkiXV3102E6p/view?usp=sharing"


def google_drive_file_id(url: str) -> str:
    m = re.search(r"/d/([^/]+)", url)
    if not m:
        raise ValueError(f"Could not extract a Google Drive file id from: {url}")
    return m.group(1)


def download_drive_file(url: str, output_name: str) -> str:
    file_id = google_drive_file_id(url)
    download_url = f"https://drive.google.com/uc?id={file_id}"
    gdown.download(download_url, output_name, quiet=False)
    return output_name


keepership_path = download_drive_file(KEEPERSHIP_URL, "ev_keepership.csv")
charger_path = download_drive_file(CHARGER_URL, "ev_chargers.csv")

keepership_raw = pd.read_csv(keepership_path, encoding="utf-8-sig")
charger_raw = pd.read_csv(charger_path, encoding="utf-8-sig")

print("EV keepership:", keepership_raw.shape)
print("EV chargers:", charger_raw.shape)

display(keepership_raw.head(3))
display(charger_raw.head(3))


# 2. Cleaning and quarterly extraction helpers

In [ ]:
WELSH_LAD_NAMES = {
    "isle of anglesey", "gwynedd", "conwy", "denbighshire", "flintshire", "wrexham",
    "powys", "ceredigion", "pembrokeshire", "carmarthenshire", "swansea", "neath port talbot",
    "bridgend", "vale of glamorgan", "cardiff", "rhondda cynon taf", "merthyr tydfil",
    "caerphilly", "blaenau gwent", "torfaen", "monmouthshire", "newport"
}

WELSH_LAD_NORMALISATION = {
    "sir ynys mon": "isle of anglesey",
    "ynys mon": "isle of anglesey",
    "isle of anglesey": "isle of anglesey",
    "sir y fflint": "flintshire",
    "caerdydd": "cardiff",
    "abertawe": "swansea",
    "casnewydd": "newport",
    "wrecsam": "wrexham",
    "rhondda cynon taff": "rhondda cynon taf",
    "the vale of glamorgan": "vale of glamorgan",
    "vale of glamorgan council": "vale of glamorgan",
    "cardiff council": "cardiff",
    "newport council": "newport",
}

WELSH_LAD_CODE_TO_NAME = {
    "W06000001": "isle of anglesey",
    "W06000002": "gwynedd",
    "W06000003": "conwy",
    "W06000004": "denbighshire",
    "W06000005": "flintshire",
    "W06000006": "wrexham",
    "W06000008": "ceredigion",
    "W06000009": "pembrokeshire",
    "W06000010": "carmarthenshire",
    "W06000011": "swansea",
    "W06000012": "neath port talbot",
    "W06000013": "bridgend",
    "W06000014": "vale of glamorgan",
    "W06000015": "cardiff",
    "W06000016": "rhondda cynon taf",
    "W06000018": "caerphilly",
    "W06000019": "blaenau gwent",
    "W06000020": "torfaen",
    "W06000021": "monmouthshire",
    "W06000022": "newport",
    "W06000023": "powys",
    "W06000024": "merthyr tydfil",
}

WELSH_LAD_NAME_TO_CODE = {v: k for k, v in WELSH_LAD_CODE_TO_NAME.items()}

quarter_order = {"Q1": 1, "Q2": 2, "Q3": 3, "Q4": 4}


def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = (
        out.columns.astype(str)
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
        .str.lower()
        .str.replace("_", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
    )
    empty_unnamed = [c for c in out.columns if c.startswith("unnamed") and out[c].isna().all()]
    return out.drop(columns=empty_unnamed, errors="ignore")


def normalise_text(value) -> str:
    if pd.isna(value):
        return ""
    text = str(value).strip().lower()
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    text = text.replace("&", "and")
    text = re.sub(r"\s+", " ", text)
    text = re.sub(
        r"\b(county borough|county|city and county|council|local authority|district|borough)\b",
        "",
        text,
    ).strip()
    text = re.sub(r"\s+", " ", text)
    return WELSH_LAD_NORMALISATION.get(text, text)


def normalise_lad_from_code_or_name(value) -> str:
    if pd.isna(value):
        return ""
    raw = str(value).strip().upper()
    if raw in WELSH_LAD_CODE_TO_NAME:
        return WELSH_LAD_CODE_TO_NAME[raw]
    return normalise_text(value)


def safe_uri_part(value) -> str:
    text = normalise_lad_from_code_or_name(value)
    text = re.sub(r"[^a-z0-9]+", "-", text).strip("-")
    return quote(text or "unknown")


def uri_fragment(value) -> str:
    text = normalise_lad_from_code_or_name(value)
    text = re.sub(r"[^a-z0-9]+", "_", text).strip("_")
    if not text:
        return "Unknown"
    return quote("_".join(part.capitalize() for part in text.split("_")))


def first_existing_col(df, candidates):
    return next((c for c in candidates if c in df.columns), None)


def looks_like_lad_code_series(s: pd.Series) -> bool:
    vals = s.dropna().astype(str).str.strip().str.upper()
    if vals.empty:
        return False
    return vals.str.match(r"^W\d{8}$").mean() > 0.5


def welsh_lad_hit_count(s: pd.Series) -> int:
    return int(s.map(normalise_lad_from_code_or_name).isin(WELSH_LAD_NAMES).sum())


def detect_lad_name_col(df: pd.DataFrame, label: str):
    name_candidates = [
        "lad name", "lad_name", "local authority name", "local authority district name",
        "local authority", "local authority district", "ons geography", "geography name",
        "geography", "area name", "area", "name", "la name"
    ]
    for c in name_candidates:
        if c in df.columns and not looks_like_lad_code_series(df[c]) and welsh_lad_hit_count(df[c]) > 0:
            return c

    best_col, best_hits = None, 0
    for c in df.columns:
        if df[c].dtype == "object" and not looks_like_lad_code_series(df[c]):
            hits = welsh_lad_hit_count(df[c])
            if hits > best_hits:
                best_col, best_hits = c, hits

    if best_col and best_hits > 0:
        return best_col

    raise ValueError(f"Could not detect LAD name column in {label}. Columns: {df.columns.tolist()}")


def detect_lad_code_col(df: pd.DataFrame):
    candidates = [
        "lad code", "lad_code", "lad", "ons code", "local authority district code",
        "local authority code", "area code", "geography code", "code", "la code"
    ]
    for c in candidates:
        if c in df.columns and looks_like_lad_code_series(df[c]):
            return c
    for c in df.columns:
        if looks_like_lad_code_series(df[c]):
            return c
    return first_existing_col(df, [c for c in candidates if c in df.columns])


def coerce_number(s):
    return pd.to_numeric(
        s.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.strip(),
        errors="coerce",
    )


def normalise_quarter(value):
    if pd.isna(value):
        return None
    text = str(value).strip().upper()

    m = re.search(r"\bQ([1-4])\b", text)
    if m:
        return f"Q{m.group(1)}"

    m = re.search(r"\bQUARTER\s*([1-4])\b", text)
    if m:
        return f"Q{m.group(1)}"

    if re.fullmatch(r"[1-4](\.0)?", text):
        return f"Q{int(float(text))}"

    dt = pd.to_datetime(value, errors="coerce", dayfirst=True)
    if pd.notna(dt):
        return f"Q{((dt.month - 1) // 3) + 1}"

    return None


def extract_year(value):
    if pd.isna(value):
        return None

    text = str(value).strip()
    m = re.search(r"\b(20\d{2}|19\d{2})\b", text)
    if m:
        return int(m.group(1))

    dt = pd.to_datetime(value, errors="coerce", dayfirst=True)
    if pd.notna(dt):
        return int(dt.year)

    return None


def year_quarter_to_t(year, quarter):
    quarter = str(quarter).upper().strip()
    if quarter not in quarter_order:
        raise ValueError(f"Invalid quarter value: {quarter}")
    return int(year) + (quarter_order[quarter] - 1) / 4


def sort_yq(values):
    return sorted(
        values,
        key=lambda x: (
            int(str(x).split("-")[0]),
            quarter_order[str(x).split("-")[1]]
        )
    )


# 3. Standardise EV keepership to LAD-quarter

In [ ]:
keepership = clean_columns(keepership_raw)

k_lad_name_col = first_existing_col(
    keepership,
    ["ons geography", "lad name", "local authority", "area name", "geography"]
)
k_lad_code_col = first_existing_col(
    keepership,
    ["ons code", "lad code", "area code", "geography code"]
)

if k_lad_name_col is None:
    k_lad_name_col = detect_lad_name_col(keepership, "EV keepership")

quarter_cols = [
    c for c in keepership.columns
    if re.match(r"^\d{4}q[1-4]$", str(c).lower().replace(" ", ""))
]

if not quarter_cols:
    raise ValueError("No keepership quarter columns found. Expected columns such as 2025q4.")

id_vars = [c for c in [k_lad_code_col, k_lad_name_col, "fuel", "keepership"] if c and c in keepership.columns]

keepership_long = keepership.melt(
    id_vars=id_vars,
    value_vars=quarter_cols,
    var_name="year_quarter_raw",
    value_name="keepership_value",
)

if "fuel" in keepership_long.columns:
    keepership_long = keepership_long[
        keepership_long["fuel"].astype(str).str.lower().str.contains("electric|battery|plug", na=False)
    ].copy()

keepership_long["year_quarter_raw"] = (
    keepership_long["year_quarter_raw"].astype(str).str.lower().str.replace(" ", "", regex=False)
)
keepership_long["year"] = keepership_long["year_quarter_raw"].str.extract(r"(\d{4})")[0].astype(int)
keepership_long["quarter"] = "Q" + keepership_long["year_quarter_raw"].str.extract(r"q([1-4])")[0]
keepership_long["year_quarter"] = keepership_long["year"].astype(str) + "-" + keepership_long["quarter"]

keepership_long["keepership_value"] = coerce_number(keepership_long["keepership_value"])
keepership_long["lad_name"] = keepership_long[k_lad_name_col].map(normalise_lad_from_code_or_name)

if k_lad_code_col:
    keepership_long["lad_code"] = keepership_long[k_lad_code_col].astype(str).str.strip().str.upper()
else:
    keepership_long["lad_code"] = keepership_long["lad_name"].map(lambda x: WELSH_LAD_NAME_TO_CODE.get(x, safe_uri_part(x)))

keepership_long = keepership_long[keepership_long["lad_name"].isin(WELSH_LAD_NAMES)].copy()
keepership_long["quarter_num"] = keepership_long["quarter"].map(quarter_order)

keepership_std = (
    keepership_long
    .dropna(subset=["year", "quarter", "lad_name", "keepership_value"])
    .drop_duplicates(subset=["lad_code", "lad_name", "year", "quarter"], keep="first")
    [["lad_code", "lad_name", "year", "quarter", "quarter_num", "year_quarter", "keepership_value"]]
    .rename(columns={"keepership_value": "keepership"})
    .sort_values(["lad_name", "year", "quarter_num"])
    .reset_index(drop=True)
)

keepership_std["keepership"] = keepership_std["keepership"].astype(float)
keepership_std["lad_uri_id"] = keepership_std["lad_name"].map(uri_fragment)

print("Standardised quarterly keepership rows:", keepership_std.shape)
print("Years:", keepership_std["year"].min(), "-", keepership_std["year"].max())
print("Quarters:", sorted(keepership_std["quarter"].unique()))

display(keepership_std.head(30))


# 4. Standardise EV charger counts to LAD-quarter

In [ ]:
chargers = clean_columns(charger_raw)

c_lad_name_col = detect_lad_name_col(chargers, "EV chargers")
c_lad_code_col = detect_lad_code_col(chargers)

print("Detected charger LAD name column:", c_lad_name_col)
print("Detected charger LAD code column:", c_lad_code_col)

month_cols = [
    c for c in chargers.columns
    if re.match(r"^(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)-\d{2,4}$", str(c).lower().strip())
]

if not month_cols:
    year_col = first_existing_col(chargers, ["year", "calendar year", "reporting year"])
    quarter_col = first_existing_col(chargers, ["quarter", "qtr", "calendar quarter", "reporting quarter", "year quarter", "year_quarter"])
    value_col = first_existing_col(chargers, ["ev_count", "ev charger count", "charger count", "count", "value", "ev_chargers"])

    if not (year_col and quarter_col and value_col):
        raise ValueError("No charger month columns found and no long-format year/quarter/value columns detected.")

    count_long = chargers.rename(columns={year_col: "year", quarter_col: "quarter", value_col: "ev_chargers"}).copy()
    count_long["year"] = count_long["year"].map(extract_year).astype("Int64")
    count_long["quarter"] = count_long["quarter"].map(normalise_quarter)
    count_long["ev_chargers"] = coerce_number(count_long["ev_chargers"])

else:
    id_vars = [c for c in [c_lad_code_col, c_lad_name_col] if c]
    count_long = chargers.melt(
        id_vars=id_vars,
        value_vars=month_cols,
        var_name="month",
        value_name="ev_chargers",
    )

    count_long["month_clean"] = count_long["month"].astype(str).str.lower().str.strip()

    count_long["month_date"] = pd.to_datetime(count_long["month_clean"], format="%b-%y", errors="coerce")
    bad = count_long["month_date"].isna()
    count_long.loc[bad, "month_date"] = pd.to_datetime(count_long.loc[bad, "month_clean"], format="%b-%Y", errors="coerce")

    count_long["year"] = count_long["month_date"].dt.year.astype("Int64")
    count_long["quarter"] = "Q" + count_long["month_date"].dt.quarter.astype("Int64").astype(str)
    count_long["ev_chargers"] = coerce_number(count_long["ev_chargers"])

if c_lad_name_col:
    count_long["lad_name"] = count_long[c_lad_name_col].map(normalise_lad_from_code_or_name)
else:
    count_long["lad_name"] = ""

if c_lad_code_col:
    count_long["lad_code"] = count_long[c_lad_code_col].astype(str).str.strip().str.upper()
    missing_name = count_long["lad_name"].eq("") | count_long["lad_name"].isna()
    count_long.loc[missing_name, "lad_name"] = count_long.loc[missing_name, "lad_code"].map(WELSH_LAD_CODE_TO_NAME)
else:
    count_long["lad_code"] = count_long["lad_name"].map(WELSH_LAD_NAME_TO_CODE).fillna(count_long["lad_name"].map(safe_uri_part))

count_long = count_long[count_long["lad_name"].isin(WELSH_LAD_NAMES)].copy()
count_long["year_quarter"] = count_long["year"].astype(str) + "-" + count_long["quarter"].astype(str)
count_long["quarter_num"] = count_long["quarter"].map(quarter_order)

if "month_date" in count_long.columns:
    # Monthly charger counts are stock/snapshot values. Use latest month in each quarter.
    count_std = (
        count_long
        .dropna(subset=["year", "quarter", "quarter_num", "lad_name", "ev_chargers", "month_date"])
        .sort_values(["lad_code", "lad_name", "year", "quarter_num", "month_date"])
        .groupby(["lad_code", "lad_name", "year", "quarter"], as_index=False)
        .tail(1)
        [["lad_code", "lad_name", "year", "quarter", "quarter_num", "year_quarter", "ev_chargers"]]
    )
else:
    count_std = (
        count_long
        .dropna(subset=["year", "quarter", "quarter_num", "lad_name", "ev_chargers"])
        .groupby(["lad_code", "lad_name", "year", "quarter", "quarter_num", "year_quarter"], as_index=False)["ev_chargers"]
        .sum()
    )

if count_std.empty:
    print("Charger debugging sample after cleaning:")
    display(chargers.head())
    display(count_long.head())
    raise ValueError("Standardised quarterly charger table is empty. Check LAD name/code/time detection.")

count_std["year"] = count_std["year"].astype(int)
count_std["quarter_num"] = count_std["quarter_num"].astype(int)
count_std["ev_chargers"] = count_std["ev_chargers"].astype(float)
count_std["lad_uri_id"] = count_std["lad_name"].map(uri_fragment)

count_std = count_std.sort_values(["lad_name", "year", "quarter_num"]).reset_index(drop=True)

print("Standardised quarterly charger rows:", count_std.shape)
print("Years:", count_std["year"].min(), "-", count_std["year"].max())
print("Quarters:", sorted(count_std["quarter"].unique()))

display(count_std.head(30))


# 5. Merge quarterly panel and validate coverage

In [ ]:
panel_df = keepership_std.merge(
    count_std,
    on=["lad_code", "lad_name", "year", "quarter", "quarter_num", "year_quarter", "lad_uri_id"],
    how="inner",
)

panel_df = panel_df.rename(columns={"keepership": "ev_keepership"})
panel_df["ev_keepership"] = panel_df["ev_keepership"].round().astype(int)
panel_df["ev_chargers"] = panel_df["ev_chargers"].round().astype(int)
panel_df["lad_uri"] = panel_df["lad_uri_id"]
panel_df["t"] = panel_df.apply(lambda r: year_quarter_to_t(r["year"], r["quarter"]), axis=1)

panel_df = (
    panel_df
    .query("2019 <= year <= 2025")
    .drop_duplicates(subset=["lad_uri", "year", "quarter"], keep="first")
    .sort_values(["lad_name", "year", "quarter_num"])
    .reset_index(drop=True)
)

print("Panel shape:", panel_df.shape)
print("LAD count:", panel_df["lad_name"].nunique())
print("Year range:", panel_df["year"].min(), "-", panel_df["year"].max())
print("Quarter values:", sorted(panel_df["quarter"].unique()))

display(panel_df.head(30))

# Expected quarters run from the first to the last quarter actually present in the
# merged panel (the charger series starts in 2019-Q4), so a LAD is only flagged
# when it has a genuine gap rather than for quarters before the panel begins.
_first = panel_df.sort_values(["year", "quarter_num"]).iloc[0]
_last = panel_df.sort_values(["year", "quarter_num"]).iloc[-1]
expected_year_quarters = [
    f"{y}-Q{qn}"
    for y in range(int(_first["year"]), int(_last["year"]) + 1)
    for qn in range(1, 5)
    if (y, qn) >= (int(_first["year"]), int(_first["quarter_num"])) and (y, qn) <= (int(_last["year"]), int(_last["quarter_num"]))
]

coverage = (
    panel_df.groupby("lad_name")["year_quarter"]
    .apply(lambda s: sort_yq(s.unique()))
    .reset_index(name="year_quarters")
)

coverage["n_quarters"] = coverage["year_quarters"].apply(len)
coverage["complete_series"] = coverage["year_quarters"].apply(lambda yqs: set(yqs) == set(expected_year_quarters))
coverage["missing_quarters"] = coverage["year_quarters"].apply(lambda yqs: sort_yq(set(expected_year_quarters) - set(yqs)))

display(coverage)

if panel_df["lad_name"].nunique() != 22:
    print("WARNING: LAD count is not 22. Check LAD naming or KG population.")

if not coverage["complete_series"].all():
    print("WARNING: Some LADs have missing quarters.")
    display(coverage.loc[~coverage["complete_series"], ["lad_name", "n_quarters", "missing_quarters"]])

panel_df.to_csv("cleets_quarterly_panel_2019_2025.csv", index=False)
print("Saved: cleets_quarterly_panel_2019_2025.csv")


# 6. Optional: build quarterly CLEETS KG

This section creates a clean quarterly RDF knowledge graph from the standardised panel. It is optional for forecasting, because `panel_df` is already the modelling table.


In [ ]:
CLEETS = Namespace("http://w3id.org/def/cleets/")
TIME = Namespace("http://www.w3.org/2006/time#")

kg = Graph()
kg.bind("cleets", CLEETS)
kg.bind("time", TIME)
kg.bind("rdf", RDF)
kg.bind("rdfs", RDFS)
kg.bind("owl", OWL)
kg.bind("xsd", XSD)

kg.add((CLEETS[""], RDF.type, OWL.Ontology))
kg.add((CLEETS[""], RDFS.label, Literal("CLEETS quarterly EV adoption ontology")))

classes = ["EV", "LAD", "WelshLAD", "Observation", "EVKeepership", "EVChargerCount", "Time"]
for cls in classes:
    kg.add((CLEETS[cls], RDF.type, OWL.Class))
    kg.add((CLEETS[cls], RDFS.label, Literal(cls)))

kg.add((CLEETS.WelshLAD, RDFS.subClassOf, CLEETS.LAD))
kg.add((CLEETS.EVKeepership, RDFS.subClassOf, CLEETS.Observation))
kg.add((CLEETS.EVChargerCount, RDFS.subClassOf, CLEETS.Observation))
kg.add((CLEETS.Time, RDFS.subClassOf, TIME.TemporalEntity))

for prop, domain, range_, label in [
    ("forLAD", CLEETS.Observation, CLEETS.LAD, "links an observation to a LAD"),
    ("forTime", CLEETS.Observation, CLEETS.Time, "links an observation to a quarterly time entity"),
]:
    kg.add((CLEETS[prop], RDF.type, OWL.ObjectProperty))
    kg.add((CLEETS[prop], RDFS.domain, domain))
    kg.add((CLEETS[prop], RDFS.range, range_))
    kg.add((CLEETS[prop], RDFS.label, Literal(label)))

for prop, dtype in {
    "ladName": XSD.string,
    "ladCode": XSD.string,
    "year": XSD.integer,
    "quarter": XSD.string,
    "yearQuarter": XSD.string,
    "keepershipValue": XSD.float,
    "chargerCountValue": XSD.float,
}.items():
    kg.add((CLEETS[prop], RDF.type, OWL.DatatypeProperty))
    kg.add((CLEETS[prop], RDFS.range, dtype))
    kg.add((CLEETS[prop], RDFS.label, Literal(prop)))


def add_lad(row):
    u = CLEETS[f"LAD#{row['lad_uri_id']}"]
    kg.add((u, RDF.type, CLEETS.LAD))
    kg.add((u, RDF.type, CLEETS.WelshLAD))
    kg.add((u, CLEETS.ladName, Literal(str(row["lad_name"]), datatype=XSD.string)))
    kg.add((u, CLEETS.ladCode, Literal(str(row["lad_code"]), datatype=XSD.string)))
    return u


def add_time(year, quarter):
    q = str(quarter).upper().strip()
    u = CLEETS[f"Time#{int(year)}_{q}"]
    kg.add((u, RDF.type, CLEETS.Time))
    kg.add((u, RDF.type, TIME.TemporalEntity))
    kg.add((u, CLEETS.year, Literal(int(year), datatype=XSD.integer)))
    kg.add((u, CLEETS.quarter, Literal(q, datatype=XSD.string)))
    kg.add((u, CLEETS.yearQuarter, Literal(f"{int(year)}-{q}", datatype=XSD.string)))
    return u


for _, row in panel_df.iterrows():
    lad = add_lad(row)
    time = add_time(row["year"], row["quarter"])

    kobs = CLEETS[f"EVKeepership#{row['lad_uri_id']}_{int(row['year'])}_{row['quarter']}"]
    kg.add((kobs, RDF.type, CLEETS.EVKeepership))
    kg.add((kobs, CLEETS.forLAD, lad))
    kg.add((kobs, CLEETS.forTime, time))
    kg.add((kobs, CLEETS.keepershipValue, Literal(float(row["ev_keepership"]), datatype=XSD.float)))

    cobs = CLEETS[f"EVChargerCount#{row['lad_uri_id']}_{int(row['year'])}_{row['quarter']}"]
    kg.add((cobs, RDF.type, CLEETS.EVChargerCount))
    kg.add((cobs, CLEETS.forLAD, lad))
    kg.add((cobs, CLEETS.forTime, time))
    kg.add((cobs, CLEETS.chargerCountValue, Literal(float(row["ev_chargers"]), datatype=XSD.float)))

kg.serialize("cleets_quarterly_kg.ttl", format="turtle")
kg.serialize("cleets_quarterly_kg.rdf", format="xml")

print("Quarterly KG triples:", len(kg))
print("Saved: cleets_quarterly_kg.ttl and cleets_quarterly_kg.rdf")


# 6b. Total private vehicle stock per LAD-quarter (VEH0105)

Section 7 sizes the scenario carrying capacity `K` from each LAD's **latest private licensed-vehicle stock** (DfT VEH0105, all body types, fuel = Total, keepership = Private), so that the EV ceiling is the fleet that could in principle be electrified rather than an arbitrary multiple of today's EV count. This cell builds `total_vehicles_df`, which Section 7 previously assumed but never created.

Source: DfT / DVLA VEH0105 (vehicle licensing statistics data tables), Drive mirror of the download used in the paper. Published values are in thousands and are converted to vehicle counts.


In [ ]:
# ============================================================
# 6b. VEH0105 private licensed-vehicle stock -> total_vehicles_df
# ============================================================
# Drive mirror of the VEH0105 download used in the companion paper notebook.
# Replace the id with your own copy if needed, or upload veh0105.csv manually
# when prompted (Drive sharing must be "Anyone with the link").
VEH0105_URL = "https://drive.google.com/file/d/1MqF57lLua8HSEFOYV0V2lnZmy5fiKGMP/view?usp=sharing"

# Exact denominator scope (matched to the private BEV numerator of VEH0132):
VEH0105_SCOPE = {"bodytype": ("total", "all", "all body types"), "fuel": "total", "keepership": "private"}


def load_veh0105(url: str) -> pd.DataFrame:
    """Download VEH0105 (CSV or Excel) and return the raw table."""
    path = "veh0105_source"
    try:
        download_drive_file(url, path)
    except Exception as exc:  # pragma: no cover - Colab network / sharing problems
        print(f"Drive download failed ({exc}). Upload veh0105.csv manually:")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("VEH0105 is required for the vehicle-stock capacity scenarios.") from exc
        path = next(iter(uploaded))
    for reader in (lambda p: pd.read_csv(p, encoding="utf-8-sig"), lambda p: pd.read_excel(p, sheet_name=0)):
        try:
            return reader(path)
        except Exception:
            continue
    raise RuntimeError("VEH0105 could not be read as CSV or Excel.")


veh0105_raw = load_veh0105(VEH0105_URL)
veh0105 = clean_columns(veh0105_raw)
print("VEH0105 shape:", veh0105.shape)

# ---- identify columns -------------------------------------------------------
v_code_col = first_existing_col(veh0105, ["ons code", "lad code", "area code", "geography code", "code"])
v_name_col = first_existing_col(veh0105, ["ons geography", "lad name", "local authority", "area name", "geography", "name"])
v_units_col = first_existing_col(veh0105, ["units", "unit"])
for required in ("bodytype", "fuel", "keepership"):
    if required not in veh0105.columns:
        raise KeyError(f"VEH0105 is missing the '{required}' column needed for the exact scope filter. "
                       f"Columns: {veh0105.columns.tolist()[:12]}")
if v_code_col is None:
    raise KeyError("Could not identify the ONS/LAD code column in VEH0105.")

v_quarter_cols = [c for c in veh0105.columns if re.match(r"^\d{4}\s?q[1-4]$", str(c).lower())]
if not v_quarter_cols:
    raise ValueError("No quarterly columns (e.g. '2025 q4') found in VEH0105.")

# ---- exact scope: Welsh LADs, all body types, total fuel, private keepership --
scope_norm = lambda s: s.fillna("").astype(str).str.strip().str.lower().str.replace(r"\s+", " ", regex=True)
veh0105[v_code_col] = veh0105[v_code_col].astype(str).str.strip().str.upper()
mask = (
    veh0105[v_code_col].str.startswith("W06", na=False)
    & scope_norm(veh0105["bodytype"]).isin(VEH0105_SCOPE["bodytype"])
    & scope_norm(veh0105["fuel"]).eq(VEH0105_SCOPE["fuel"])
    & scope_norm(veh0105["keepership"]).eq(VEH0105_SCOPE["keepership"])
)
veh_scope = veh0105.loc[mask].copy()
if veh_scope[v_code_col].nunique() != 22 or veh_scope.duplicated(subset=[v_code_col]).any():
    raise RuntimeError(
        f"Expected exactly one VEH0105 row per Welsh LAD (22) in scope bodytype=Total/All, fuel=Total, "
        f"keepership=Private; found {len(veh_scope)} rows for {veh_scope[v_code_col].nunique()} LADs. "
        f"Body types present: {sorted(scope_norm(veh0105['bodytype']).unique())}"
    )

# ---- units: VEH0105 publishes thousands of vehicles ---------------------------
def veh0105_multiplier(unit) -> float:
    text = "" if pd.isna(unit) else str(unit).strip().lower()
    if "thousand" in text or "000" in text:
        return 1000.0
    if text in {"", "number", "numbers", "count", "counts", "vehicles", "units"}:
        return 1.0
    raise ValueError(f"Unrecognised VEH0105 units label: {unit!r}")

veh_scope["unit_multiplier"] = veh_scope[v_units_col].map(veh0105_multiplier) if v_units_col else 1.0

# ---- reshape to LAD-quarter --------------------------------------------------
id_vars = [c for c in [v_code_col, v_name_col, "unit_multiplier"] if c]
stock_long = veh_scope.melt(id_vars=id_vars, value_vars=v_quarter_cols,
                            var_name="year_quarter_raw", value_name="published_value")
stock_long["published_value"] = coerce_number(stock_long["published_value"])
stock_long = stock_long.dropna(subset=["published_value"])
stock_long["total_vehicle_stock"] = (stock_long["published_value"] * stock_long["unit_multiplier"]).round().astype(int)

yq = stock_long["year_quarter_raw"].astype(str).str.lower().str.replace(" ", "", regex=False)
stock_long["year"] = yq.str.extract(r"(\d{4})")[0].astype(int)
stock_long["quarter"] = "Q" + yq.str.extract(r"q([1-4])")[0]
stock_long["quarter_num"] = stock_long["quarter"].map(quarter_order)
stock_long["year_quarter"] = stock_long["year"].astype(str) + "-" + stock_long["quarter"]
stock_long["lad_code"] = stock_long[v_code_col]
stock_long["lad_name"] = stock_long["lad_code"].map(WELSH_LAD_CODE_TO_NAME)
stock_long["lad_uri"] = stock_long["lad_name"].map(uri_fragment)

total_vehicles_df = (
    stock_long
    .dropna(subset=["lad_name"])
    [["lad_code", "lad_name", "lad_uri", "year", "quarter", "quarter_num", "year_quarter", "total_vehicle_stock"]]
    .sort_values(["lad_name", "year", "quarter_num"])
    .reset_index(drop=True)
)

# ---- plausibility check (private share of Cardiff's total licensed fleet) ------
latest_yq = total_vehicles_df.sort_values(["year", "quarter_num"]).iloc[-1]["year_quarter"]
cardiff_private = int(total_vehicles_df.query("lad_code == 'W06000015' and year_quarter == @latest_yq")["total_vehicle_stock"].iloc[0])
total_mask = (
    veh0105[v_code_col].eq("W06000015")
    & scope_norm(veh0105["bodytype"]).isin(VEH0105_SCOPE["bodytype"])
    & scope_norm(veh0105["fuel"]).eq("total")
    & scope_norm(veh0105["keepership"]).eq("total")
)
latest_col = [c for c in v_quarter_cols if c.lower().replace(" ", "") == latest_yq.replace("-", "").lower()][0]
if total_mask.any():
    cardiff_total = float(coerce_number(veh0105.loc[total_mask, latest_col]).iloc[0]) * (
        veh0105_multiplier(veh0105.loc[total_mask, v_units_col].iloc[0]) if v_units_col else 1.0)
    share = cardiff_private / cardiff_total
    print(f"Cardiff {latest_yq}: private {cardiff_private:,} of {cardiff_total:,.0f} licensed vehicles ({share:.1%} private)")
    if not 0.60 <= share <= 0.95:
        raise AssertionError("Cardiff's private share of the licensed fleet is outside 60-95%: the VEH0105 scope or units are wrong.")

print("total_vehicles_df:", total_vehicles_df.shape,
      "| LADs:", total_vehicles_df["lad_name"].nunique(),
      "| quarters:", total_vehicles_df["year_quarter"].iloc[0], "to", latest_yq)
display(total_vehicles_df.query("year_quarter == @latest_yq").sort_values("total_vehicle_stock", ascending=False).head(22))

total_vehicles_df.to_csv("veh0105_wales_lad_quarter_private_vehicle_stock.csv", index=False)
print("Saved: veh0105_wales_lad_quarter_private_vehicle_stock.csv")


# 7. Capacity scenarios

The bounded logistic model uses a scenario-specific carrying capacity `K`. Instead of an arbitrary multiple of today's EV count, `K` for EV keepership is a **target share of the LAD's latest private licensed-vehicle stock** (VEH0105, Section 6b): 80% (low), 100% (central) and 110% (high, allowing fleet growth). The charger capacity follows from the EV capacity through an assumed EVs-per-public-charger ratio: 30 (low), 20 (central) and 15 (high). Both are editable scenario assumptions, not forecasts; the resulting table is saved so it can be revised.


In [ ]:
# latest EV + charger counts by LAD
latest_counts = (
    panel_df
    .sort_values(["lad_name", "year", "quarter_num"])
    .groupby(["lad_uri", "lad_name"], as_index=False)
    .tail(1)
    [[
        "lad_uri", "lad_name", "year", "quarter", "year_quarter",
        "ev_keepership", "ev_chargers"
    ]]
    .reset_index(drop=True)
)

# total vehicles by LAD
# IMPORTANT: total_vehicle_stock must be the latest total number of vehicles
# for each LAD, from VEH0105 or your cleaned total-vehicles file.
latest_vehicle_stock = (
    total_vehicles_df
    .sort_values(["lad_name", "year", "quarter_num"])
    .groupby(["lad_uri", "lad_name"], as_index=False)
    .tail(1)
    [["lad_uri", "lad_name", "total_vehicle_stock"]]
    .reset_index(drop=True)
)

latest_counts = latest_counts.merge(
    latest_vehicle_stock,
    on=["lad_uri", "lad_name"],
    how="left"
)

scenario_capacity_share = {
    "low": 0.80,       # 80% EV adoption
    "central": 1.00,   # 100% EV adoption
    "high": 1.10       # allows fleet growth / higher future stock
}

charger_ratios = {
    "low": 30,       # 1 charger per 30 EVs
    "central": 20,   # 1 charger per 20 EVs
    "high": 15       # 1 charger per 15 EVs
}

capacity_rows = []

for _, row in latest_counts.iterrows():

    latest_ev = int(row["ev_keepership"])
    latest_chargers = int(row["ev_chargers"])
    total_stock = int(row["total_vehicle_stock"])

    for scenario, share in scenario_capacity_share.items():

        K_ev = int(round(total_stock * share))

        capacity_rows.append({
            "scenario": scenario,
            "lad_uri": row["lad_uri"],
            "lad_name": row["lad_name"],
            "latest_year": int(row["year"]),
            "latest_quarter": row["quarter"],
            "latest_year_quarter": row["year_quarter"],

            "latest_ev_keepership": latest_ev,
            "latest_ev_chargers": latest_chargers,
            "total_vehicle_stock": total_stock,

            # EV capacity based on total vehicles, not arbitrary multiplier
            "K_ev_keepership": max(latest_ev + 1, K_ev),

            # charger capacity derived from EV capacity
            "K_ev_chargers": max(
                latest_chargers + 1,
                int(round(K_ev / charger_ratios[scenario]))
            ),

            "target_adoption_share": share,
            "charger_ratio_ev_per_charger": charger_ratios[scenario]
        })

capacity_df = pd.DataFrame(capacity_rows)

display(capacity_df)
display(
    capacity_df
    .groupby("scenario")[["K_ev_keepership", "K_ev_chargers"]]
    .mean()
)

capacity_df.to_csv(
    "editable_lad_capacity_scenarios_quarterly_vehicle_stock_based.csv",
    index=False
)

print("Saved: editable_lad_capacity_scenarios_quarterly_vehicle_stock_based.csv")

# 8. Forecasting models and metrics

The notebook compares several stable models. This is more reliable than choosing one model by assumption.

Models:
- `naive_last`: repeats the last observed value.
- `linear_log`: linear trend on `log1p(y)`.
- `quadratic_log`: quadratic trend on `log1p(y)`.
- `bounded_logistic_scenario`: bounded logistic curve using scenario-specific `K`.
- `ensemble_best`: average of the two best non-scenario models from backtesting for each target.


In [ ]:
def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

    if len(y_true) > 1:
        ss_res = np.sum((y_true - y_pred) ** 2)
        ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
        r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else np.nan
    else:
        r2 = np.nan

    return mae, rmse, r2


def logistic_curve(t, K, r, t0):
    z = np.clip(-r * (np.asarray(t, dtype=float) - t0), -700, 700)
    return K / (1.0 + np.exp(z))


def predict_naive(time_values, values, forecast_time_values):
    value = float(np.asarray(values, dtype=float)[-1])
    return np.full(len(forecast_time_values), value)


def predict_poly_log(time_values, values, forecast_time_values, degree=1):
    t = np.asarray(time_values, dtype=float)
    y = np.asarray(values, dtype=float)
    tf = np.asarray(forecast_time_values, dtype=float)

    degree = min(degree, len(t) - 1)
    if degree < 1:
        return predict_naive(t, y, tf)

    y_log = np.log1p(np.clip(y, 0, None))
    coefs = np.polyfit(t, y_log, degree)
    pred_log = np.polyval(coefs, tf)
    pred = np.expm1(pred_log)
    return np.clip(pred, 0, None)


def fit_bounded_logistic(time_values, values, K, forecast_time_values):
    t = np.asarray(time_values, dtype=float)
    y = np.asarray(values, dtype=float)
    tf = np.asarray(forecast_time_values, dtype=float)
    K_input = float(K)

    finite = np.isfinite(t) & np.isfinite(y)
    t = t[finite]
    y = y[finite]

    if len(t) < 3:
        return predict_naive(t, y, tf), np.nan, np.nan, K_input, "fallback_naive_too_few"

    observed_max = np.nanmax(y)
    K_used = max(K_input, observed_max * 1.05, observed_max + 1)

    y_fit = np.clip(y, 1, K_used - 1)

    try:
        y_logit = np.log(y_fit / (K_used - y_fit))
        r, intercept = np.polyfit(t, y_logit, 1)

        if not np.isfinite(r) or r <= 0 or abs(r) < 1e-8:
            raise ValueError("Invalid growth rate.")

        t0 = -intercept / r
        pred = logistic_curve(tf, K_used, r, t0)
        status = "scenario_K_logistic_fit"

    except Exception as e:
        # Fallback: monotone log-linear trend capped at K.
        pred = predict_poly_log(t, y, tf, degree=1)
        r, t0 = np.nan, np.nan
        status = f"fallback_log_linear: {str(e)[:80]}"

    pred = np.clip(pred, 0, K_used)
    return pred, float(r) if np.isfinite(r) else np.nan, float(t0) if np.isfinite(t0) else np.nan, float(K_used), status


def run_model(model_name, train_t, train_y, forecast_t, K=None):
    if model_name == "naive_last":
        pred = predict_naive(train_t, train_y, forecast_t)
        return pred, {}

    if model_name == "linear_log":
        pred = predict_poly_log(train_t, train_y, forecast_t, degree=1)
        return pred, {}

    if model_name == "quadratic_log":
        pred = predict_poly_log(train_t, train_y, forecast_t, degree=2)
        return pred, {}

    if model_name == "bounded_logistic_scenario":
        pred, r, t0, K_used, status = fit_bounded_logistic(train_t, train_y, K, forecast_t)
        return pred, {"r": r, "t0": t0, "K_used": K_used, "fit_status": status}

    raise ValueError(f"Unknown model: {model_name}")


def summarise_metrics(metrics_df):
    return (
        metrics_df
        .groupby(["model", "scenario", "target"], as_index=False)
        .agg(
            mean_MAE=("MAE", "mean"),
            mean_RMSE=("RMSE", "mean"),
            mean_R2=("R2", "mean"),
            n_LADs=("lad_name", "nunique"),
            n_test_predictions=("n_test_predictions", "sum"),
        )
        .sort_values(["target", "mean_RMSE", "mean_MAE"])
        .reset_index(drop=True)
    )


# 9. Quarterly walk-forward backtesting

This section evaluates models on 2023–2025 using only past quarters available at each prediction point. The best model is selected by lowest mean RMSE.


In [ ]:
test_years = [2023, 2024, 2025]
test_quarters = ["Q1", "Q2", "Q3", "Q4"]

test_periods = [
    {
        "year": y,
        "quarter": q,
        "year_quarter": f"{y}-{q}",
        "t": year_quarter_to_t(y, q),
    }
    for y in test_years
    for q in test_quarters
]

candidate_models = ["naive_last", "linear_log", "quadratic_log", "bounded_logistic_scenario"]
targets = ["ev_keepership", "ev_chargers"]

backtest_rows = []

for target in targets:
    K_col = "K_ev_keepership" if target == "ev_keepership" else "K_ev_chargers"

    for scenario in ["low", "central", "high"]:
        cap_s = capacity_df[capacity_df["scenario"] == scenario].set_index("lad_uri")

        for (lad_uri, lad_name), part in panel_df.groupby(["lad_uri", "lad_name"]):
            part = part.dropna(subset=["t", target]).sort_values("t").copy()

            if lad_uri not in cap_s.index:
                continue

            K_value = float(cap_s.loc[lad_uri, K_col])

            for model_name in candidate_models:
                for test_period in test_periods:
                    test_t = test_period["t"]
                    test_yq = test_period["year_quarter"]

                    train = part[part["t"] < test_t]
                    test = part[part["year_quarter"] == test_yq]

                    if len(train) < 3 or test.empty:
                        continue

                    pred, info = run_model(
                        model_name=model_name,
                        train_t=train["t"].values,
                        train_y=train[target].values,
                        forecast_t=[test_t],
                        K=K_value,
                    )

                    backtest_rows.append({
                        "model": model_name,
                        "scenario": scenario,
                        "target": target,
                        "lad_uri": lad_uri,
                        "lad_name": lad_name,
                        "test_year_quarter": test_yq,
                        "actual": float(test[target].iloc[0]),
                        "predicted": float(np.asarray(pred).ravel()[0]),
                        "K_input": K_value,
                        **info,
                    })

backtest_predictions_df = pd.DataFrame(backtest_rows)

if backtest_predictions_df.empty:
    raise ValueError("No backtest predictions were produced. Check panel_df and capacity_df.")

metric_rows = []

for (model, scenario, target, lad_uri, lad_name), group in backtest_predictions_df.groupby(["model", "scenario", "target", "lad_uri", "lad_name"]):
    mae, rmse, r2 = compute_metrics(group["actual"].values, group["predicted"].values)
    metric_rows.append({
        "model": model,
        "scenario": scenario,
        "target": target,
        "lad_uri": lad_uri,
        "lad_name": lad_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "n_test_predictions": len(group),
    })

model_metrics_df = pd.DataFrame(metric_rows)
model_summary_df = summarise_metrics(model_metrics_df)

display(model_summary_df)
display(model_metrics_df.sort_values(["target", "lad_name", "scenario", "RMSE"]).head(40))

backtest_predictions_df.to_csv("model_backtest_predictions_quarterly_2023_2025.csv", index=False)
model_metrics_df.to_csv("model_backtest_lad_level_metrics.csv", index=False)
model_summary_df.to_csv("model_backtest_summary_metrics.csv", index=False)

print("Saved:")
print("- model_backtest_predictions_quarterly_2023_2025.csv")
print("- model_backtest_lad_level_metrics.csv")
print("- model_backtest_summary_metrics.csv")


# 10. Select the best model

For short-term accuracy, we select the model with lowest RMSE for each target. Scenario rows for non-scenario models are identical by design; the bounded logistic model is kept for long-term scenario forecasting because it uses `K`.


In [ ]:
# Best model per target/scenario
best_by_target_scenario = (
    model_summary_df
    .sort_values(["target", "scenario", "mean_RMSE", "mean_MAE"])
    .groupby(["target", "scenario"], as_index=False)
    .first()
)

display(best_by_target_scenario)

# Best non-scenario models per target for accuracy ensemble
non_scenario_summary = model_summary_df[
    model_summary_df["model"].isin(["naive_last", "linear_log", "quadratic_log"])
].copy()

best_non_scenario_models = (
    non_scenario_summary
    .groupby(["target", "model"], as_index=False)
    .agg(mean_RMSE=("mean_RMSE", "mean"))
    .sort_values(["target", "mean_RMSE"])
)

top2_models_by_target = (
    best_non_scenario_models
    .groupby("target")
    .head(2)
    .groupby("target")["model"]
    .apply(list)
    .to_dict()
)

print("Top two non-scenario models by target:")
print(top2_models_by_target)


# 11. Forecast to 2045 with corrected hybrid scenario method

To improve practical accuracy, the final forecast uses a hybrid:

- Near-term behaviour is guided by the best-performing non-scenario trend models from backtesting.
- Long-term scenario divergence is driven by bounded logistic `K`.
- A smooth blending weight increases from 0 in 2026 to 1 in 2045.

This avoids unrealistic early jumps while still allowing low/central/high scenarios to diverge by 2045.


In [ ]:
forecast_years = list(range(2026, 2046))
forecast_quarters = ["Q1", "Q2", "Q3", "Q4"]

forecast_periods = [
    {
        "year": y,
        "quarter": q,
        "year_quarter": f"{y}-{q}",
        "t": year_quarter_to_t(y, q),
    }
    for y in forecast_years
    for q in forecast_quarters
]

forecast_t = np.array([p["t"] for p in forecast_periods], dtype=float)
blend_weight = np.linspace(0.0, 1.0, len(forecast_t))  # 0 = accuracy trend, 1 = scenario logistic

forecast_rows = []
parameter_rows = []

for target in targets:
    K_col = "K_ev_keepership" if target == "ev_keepership" else "K_ev_chargers"
    top_models = top2_models_by_target.get(target, ["linear_log", "quadratic_log"])

    for scenario in ["low", "central", "high"]:
        cap_s = capacity_df[capacity_df["scenario"] == scenario].set_index("lad_uri")

        for (lad_uri, lad_name), part in panel_df.groupby(["lad_uri", "lad_name"]):
            part = part.dropna(subset=["t", target]).sort_values("t").copy()

            if len(part) < 3 or lad_uri not in cap_s.index:
                continue

            K_value = float(cap_s.loc[lad_uri, K_col])

            # Accuracy component: average of top two non-scenario models.
            acc_preds = []
            for m in top_models:
                pred, _ = run_model(
                    model_name=m,
                    train_t=part["t"].values,
                    train_y=part[target].values,
                    forecast_t=forecast_t,
                    K=None,
                )
                acc_preds.append(pred)

            accuracy_pred = np.mean(np.vstack(acc_preds), axis=0)

            # Scenario component: bounded logistic with K.
            logistic_pred, info = run_model(
                model_name="bounded_logistic_scenario",
                train_t=part["t"].values,
                train_y=part[target].values,
                forecast_t=forecast_t,
                K=K_value,
            )

            # Hybrid forecast: trend near 2026, scenario K near 2045.
            hybrid_pred = (1 - blend_weight) * accuracy_pred + blend_weight * logistic_pred
            hybrid_pred = np.clip(hybrid_pred, 0, info.get("K_used", K_value))
            hybrid_pred = np.rint(hybrid_pred).astype(int)

            parameter_rows.append({
                "target": target,
                "scenario": scenario,
                "lad_uri": lad_uri,
                "lad_name": lad_name,
                "K_input": K_value,
                "K_used": info.get("K_used", np.nan),
                "r": info.get("r", np.nan),
                "t0": info.get("t0", np.nan),
                "fit_status": info.get("fit_status", ""),
                "accuracy_models_used": ", ".join(top_models),
            })

            for period, yhat in zip(forecast_periods, hybrid_pred):
                forecast_rows.append({
                    "model": "hybrid_accuracy_plus_bounded_logistic",
                    "scenario": scenario,
                    "target": target,
                    "lad_uri": lad_uri,
                    "lad_name": lad_name,
                    "year": int(period["year"]),
                    "quarter": period["quarter"],
                    "year_quarter": period["year_quarter"],
                    "t": period["t"],
                    "prediction": int(yhat),
                    "K_input": K_value,
                    "K_used": info.get("K_used", np.nan),
                })

final_forecast_long_df = pd.DataFrame(forecast_rows)
forecast_parameters_df = pd.DataFrame(parameter_rows)

display(final_forecast_long_df.head(30))
display(forecast_parameters_df.head(30))

final_forecast_long_df.to_csv("final_hybrid_quarterly_forecasts_2026_2045_long.csv", index=False)
forecast_parameters_df.to_csv("final_hybrid_quarterly_forecast_parameters.csv", index=False)

print("Saved:")
print("- final_hybrid_quarterly_forecasts_2026_2045_long.csv")
print("- final_hybrid_quarterly_forecast_parameters.csv")


# 12. Wide forecast tables for EV keepership and chargers

In [ ]:
ev_keepership_predictions_2045_df = (
    final_forecast_long_df
    .query("target == 'ev_keepership'")
    .rename(columns={"prediction": "predicted_ev_keepership"})
    [["model", "scenario", "lad_uri", "lad_name", "year", "quarter", "year_quarter", "predicted_ev_keepership", "K_input", "K_used"]]
    .sort_values(["scenario", "lad_name", "year", "quarter"])
    .reset_index(drop=True)
)

ev_charger_predictions_2045_df = (
    final_forecast_long_df
    .query("target == 'ev_chargers'")
    .rename(columns={"prediction": "predicted_ev_chargers"})
    [["model", "scenario", "lad_uri", "lad_name", "year", "quarter", "year_quarter", "predicted_ev_chargers", "K_input", "K_used"]]
    .sort_values(["scenario", "lad_name", "year", "quarter"])
    .reset_index(drop=True)
)

display(ev_keepership_predictions_2045_df.head(30))
display(ev_charger_predictions_2045_df.head(30))

ev_keepership_predictions_2045_df.to_csv("ev_keepership_predictions_to_2045_corrected.csv", index=False)
ev_charger_predictions_2045_df.to_csv("ev_charger_predictions_to_2045_corrected.csv", index=False)

print("Saved:")
print("- ev_keepership_predictions_to_2045_corrected.csv")
print("- ev_charger_predictions_to_2045_corrected.csv")


# 13. Scenario comparison and plots

In [ ]:
scenario_totals = (
    final_forecast_long_df
    .groupby(["scenario", "target", "year", "quarter", "year_quarter"], as_index=False)["prediction"]
    .sum()
)

display(
    scenario_totals
    .query("year == 2045 and quarter == 'Q4'")
    .sort_values(["target", "scenario"])
)

fig = px.line(
    scenario_totals,
    x="year_quarter",
    y="prediction",
    color="scenario",
    facet_row="target",
    title="Wales total quarterly EV forecasts by scenario, 2026–2045",
    markers=False,
    height=700,
    labels={"prediction": "Predicted total", "year_quarter": "Quarter"},
)

fig.update_yaxes(matches=None)
fig.update_xaxes(tickangle=45)
fig.for_each_annotation(lambda a: a.update(text=a.text.replace("target=", "")))
fig.show()


# 14. Final recommendation table

This table states which model performed best in historical backtesting and which model is used for final scenario forecasting.


In [ ]:
recommendation_rows = []

for target in targets:
    best = (
        model_summary_df
        .query("target == @target")
        .sort_values(["mean_RMSE", "mean_MAE"])
        .iloc[0]
    )

    recommendation_rows.append({
        "target": target,
        "best_backtest_model": best["model"],
        "best_backtest_scenario": best["scenario"],
        "mean_RMSE": best["mean_RMSE"],
        "mean_MAE": best["mean_MAE"],
        "mean_R2": best["mean_R2"],
        "final_forecast_model": "hybrid_accuracy_plus_bounded_logistic",
        "reason": "Uses the strongest backtested short-term trend and scenario K for long-term 2045 divergence."
    })

model_recommendation_df = pd.DataFrame(recommendation_rows)

display(model_recommendation_df)

model_recommendation_df.to_csv("model_recommendation_summary.csv", index=False)
print("Saved: model_recommendation_summary.csv")


# 15. Download outputs

In [ ]:
files.download("model_backtest_summary_metrics.csv")
files.download("model_backtest_lad_level_metrics.csv")
files.download("model_recommendation_summary.csv")
files.download("final_hybrid_quarterly_forecasts_2026_2045_long.csv")
files.download("ev_keepership_predictions_to_2045_corrected.csv")
files.download("ev_charger_predictions_to_2045_corrected.csv")
